# 🗄️ LAB D9 – Administración y Almacenamiento de BD
## Fundamentos de Gestión de Datos | TECSUP 2026-I
**Docente:** Mg. Pilar Rocío Sayán Mejía  
**Semana:** 9 | **Unidad:** III — Almacenamiento + ETL + Metadatos


## 🎯 Capacidad a Lograr
Gestionar registros aplicando DML (INSERT, UPDATE, DELETE) con control de transacciones COMMIT/ROLLBACK en SQLite desde Google Colab, con manejo de errores y tabla de auditoría.


---
## ⚙️ Paso 1: Conexión y diagnóstico inicial de la BD


In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime

print("✅ Librerías listas")


In [ ]:
# Conectar y preparar la base de datos del proyecto
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente   INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre       TEXT NOT NULL,
    email        TEXT UNIQUE NOT NULL,
    ciudad       TEXT,
    activo       INTEGER DEFAULT 1
);
CREATE TABLE IF NOT EXISTS productos (
    id_producto  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre       TEXT NOT NULL,
    categoria    TEXT,
    precio       REAL CHECK(precio > 0),
    stock        INTEGER DEFAULT 0
);
CREATE TABLE IF NOT EXISTS ventas (
    id_venta     INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente   INTEGER NOT NULL REFERENCES clientes(id_cliente),
    id_producto  INTEGER NOT NULL REFERENCES productos(id_producto),
    cantidad     INTEGER NOT NULL CHECK(cantidad > 0),
    monto_total  REAL,
    fecha_venta  TEXT DEFAULT CURRENT_DATE
);
CREATE TABLE IF NOT EXISTS log_operaciones (
    id_log       INTEGER PRIMARY KEY AUTOINCREMENT,
    tabla        TEXT,
    operacion    TEXT,
    filas_afect  INTEGER,
    usuario      TEXT DEFAULT 'estudiante',
    fecha_hora   TEXT DEFAULT CURRENT_TIMESTAMP,
    descripcion  TEXT
);
""")

# Insertar datos iniciales
conn.executescript("""
INSERT INTO clientes (nombre, email, ciudad) VALUES
  ('Ana Torres','ana@mail.com','Lima'),
  ('Luis Ríos','luis@mail.com','Arequipa'),
  ('María Paz','maria@mail.com','Cusco'),
  ('Carlos Vega','carlos@mail.com','Lima'),
  ('Rosa Llanos','rosa@mail.com','Trujillo');
INSERT INTO productos (nombre, categoria, precio, stock) VALUES
  ('Laptop HP 15','Tecnología',2800.0,20),
  ('Mouse Inalámbrico','Accesorios',45.0,150),
  ('Teclado Mecánico','Accesorios',180.0,80),
  ('Monitor 24"','Tecnología',650.0,35),
  ('USB-C Hub','Accesorios',85.0,200);
INSERT INTO ventas (id_cliente, id_producto, cantidad, monto_total) VALUES
  (1,1,1,2800.0),(2,2,3,135.0),(3,3,1,180.0),(4,4,2,1300.0),(5,5,5,425.0);
""")
conn.commit()

# Diagnóstico inicial
print("=" * 55)
print("ESTADO INICIAL DE LA BASE DE DATOS")
print("=" * 55)
for tabla in ['clientes','productos','ventas','log_operaciones']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla:20s}: {n:3d} registros")


## ➕ Paso 2: Operaciones INSERT — Inserción de nuevos registros


In [ ]:
# PASO 2: INSERT — individual y múltiple
print("=== OPERACIONES INSERT ===\n")

def log_op(tabla, operacion, filas, descripcion):
    conn.execute(
        "INSERT INTO log_operaciones (tabla,operacion,filas_afect,descripcion) VALUES (?,?,?,?)",
        (tabla, operacion, filas, descripcion)
    )

# Inserción individual
cursor = conn.cursor()
cursor.execute(
    "INSERT INTO clientes (nombre, email, ciudad) VALUES (?,?,?)",
    ('Diego Quispe', 'diego@mail.com', 'Piura')
)
nuevo_id = cursor.lastrowid
conn.commit()
log_op('clientes','INSERT',1,f"Nuevo cliente: Diego Quispe (id={nuevo_id})")
print(f"✅ INSERT individual: id_cliente = {nuevo_id}")

# Inserción múltiple con executemany
nuevos_productos = [
    ('Webcam HD', 'Accesorios', 120.0, 60),
    ('Silla Ergonómica', 'Mobiliario', 450.0, 15),
    ('Auriculares BT', 'Accesorios', 95.0, 100),
]
cursor.executemany(
    "INSERT INTO productos (nombre, categoria, precio, stock) VALUES (?,?,?,?)",
    nuevos_productos
)
filas = cursor.rowcount
conn.commit()
log_op('productos','INSERT',filas,f"Carga masiva de {filas} nuevos productos")
print(f"✅ INSERT múltiple: {filas} productos insertados")

# Verificar
print("\n📋 Clientes actuales:")
print(pd.read_sql_query("SELECT id_cliente,nombre,ciudad FROM clientes",conn).to_string(index=False))
print(f"\n  Total productos: {conn.execute('SELECT COUNT(*) FROM productos').fetchone()[0]}")


## ✏️ Paso 3: Operaciones UPDATE — Actualización de registros


In [ ]:
# PASO 3: UPDATE — directo y con subquery
print("=== OPERACIONES UPDATE ===\n")

# UPDATE directo: actualizar precio con cláusula WHERE
cursor.execute("""
    UPDATE productos
    SET precio = precio * 1.08
    WHERE categoria = 'Tecnología'
""")
filas_afect = cursor.rowcount
conn.commit()
log_op('productos','UPDATE',filas_afect,"Incremento 8% en precios de Tecnología")
print(f"✅ UPDATE precios Tecnología: {filas_afect} productos actualizados")

# UPDATE con subquery: actualizar ciudad del cliente con más ventas
cursor.execute("""
    UPDATE clientes
    SET ciudad = 'Lima - Sede Principal'
    WHERE id_cliente = (
        SELECT id_cliente FROM ventas
        GROUP BY id_cliente
        ORDER BY COUNT(*) DESC
        LIMIT 1
    )
""")
filas2 = cursor.rowcount
conn.commit()
log_op('clientes','UPDATE',filas2,"Ciudad actualizada al cliente con más ventas")
print(f"✅ UPDATE con subquery: {filas2} cliente actualizado")

# UPDATE masivo: marcar como inactivos clientes sin ventas
cursor.execute("""
    UPDATE clientes
    SET activo = 0
    WHERE id_cliente NOT IN (SELECT DISTINCT id_cliente FROM ventas)
""")
filas3 = cursor.rowcount
conn.commit()
log_op('clientes','UPDATE',filas3,"Inactivación de clientes sin ventas")
print(f"✅ UPDATE masivo: {filas3} clientes marcados como inactivos")

print("\n📋 Precios actualizados (Tecnología):")
print(pd.read_sql_query(
    "SELECT nombre,precio FROM productos WHERE categoria='Tecnología'",conn).to_string(index=False))


## 🗑️ Paso 4: Operaciones DELETE y control con ROLLBACK


In [ ]:
# PASO 4: DELETE + ROLLBACK demo
print("=== DELETE + CONTROL DE TRANSACCIONES ===\n")

# ─── DEMO: DELETE accidental + ROLLBACK ───
print("--- DEMO: DELETE accidental ---")
conn2 = sqlite3.connect(":memory:")
conn2.execute("PRAGMA foreign_keys = ON")
conn2.execute("CREATE TABLE demo (id INTEGER, valor TEXT)")
conn2.executemany("INSERT INTO demo VALUES (?,?)", [(1,'A'),(2,'B'),(3,'C')])
conn2.commit()

n_antes = conn2.execute("SELECT COUNT(*) FROM demo").fetchone()[0]
print(f"Registros antes del DELETE accidental: {n_antes}")

# Simular DELETE sin WHERE dentro de transacción (modo manual)
conn2.isolation_level = None  # autocommit OFF
conn2.execute("BEGIN")
conn2.execute("DELETE FROM demo")  # ¡Sin WHERE!
n_durante = conn2.execute("SELECT COUNT(*) FROM demo").fetchone()[0]
print(f"Registros durante (sin commit): {n_durante}  ← datos 'perdidos' temporalmente")

conn2.execute("ROLLBACK")
n_despues = conn2.execute("SELECT COUNT(*) FROM demo").fetchone()[0]
print(f"Registros DESPUÉS del ROLLBACK : {n_despues}  ← datos recuperados ✅")
conn2.close()

# ─── DELETE correcto en BD del proyecto ───
print("\n--- DELETE correcto con WHERE ---")
cursor.execute("SELECT COUNT(*) FROM productos WHERE stock = 0")
sin_stock = cursor.fetchone()[0]
cursor.execute("DELETE FROM productos WHERE stock = 0")
filas_del = cursor.rowcount
conn.commit()
log_op('productos','DELETE',filas_del,"Eliminación de productos sin stock")
print(f"✅ DELETE correcto: {filas_del} productos con stock=0 eliminados")
print(f"   (verificado rowcount: {filas_del})")


## 🔒 Paso 5: Transacciones y manejo de errores


In [ ]:
# PASO 5: try-except-finally + integridad referencial
print("=== MANEJO DE ERRORES EN TRANSACCIONES ===\n")

# Test 1: Violación de FK (venta con cliente inexistente)
print("Test 1: Venta con id_cliente inexistente (debe fallar)")
try:
    conn.execute("BEGIN")
    conn.execute(
        "INSERT INTO ventas (id_cliente, id_producto, cantidad, monto_total) VALUES (999, 1, 1, 2800.0)"
    )
    conn.execute("COMMIT")
    print("  ⚠️ No se detectó error (PRAGMA FK puede estar desactivado)")
except sqlite3.IntegrityError as e:
    conn.execute("ROLLBACK")
    print(f"  ✅ Error capturado correctamente: {e}")
    print("  ✅ ROLLBACK ejecutado — datos intactos")
finally:
    print("  → Bloque finally ejecutado siempre")

# Test 2: Violación de CHECK (precio negativo)
print("\nTest 2: Producto con precio negativo (debe fallar)")
try:
    conn.execute("BEGIN")
    conn.execute("INSERT INTO productos (nombre,categoria,precio,stock) VALUES ('Mal Producto','X',-50.0,0)")
    conn.execute("COMMIT")
except sqlite3.IntegrityError as e:
    conn.execute("ROLLBACK")
    print(f"  ✅ Error capturado: {e}")

# Test 3: UNIQUE violation
print("\nTest 3: Email duplicado (debe fallar)")
try:
    conn.execute("BEGIN")
    conn.execute("INSERT INTO clientes (nombre,email,ciudad) VALUES ('Otro','ana@mail.com','Lima')")
    conn.execute("COMMIT")
except sqlite3.IntegrityError as e:
    conn.execute("ROLLBACK")
    print(f"  ✅ Error capturado: {e}")

print("\n✅ Todos los controles de integridad funcionan correctamente")


## 📋 Paso 6: Tabla de auditoría y reporte final


In [ ]:
# PASO 6: Auditoría completa y reporte
print("=== REGISTRO DE AUDITORÍA DML ===\n")

df_log = pd.read_sql_query(
    "SELECT tabla, operacion, filas_afect, fecha_hora, descripcion FROM log_operaciones ORDER BY id_log",
    conn
)
print(df_log.to_string(index=False))

print("\n📊 Resumen por tipo de operación:")
resumen = df_log.groupby('operacion').agg(
    n_operaciones=('operacion','count'),
    total_filas=('filas_afect','sum')
).reset_index()
print(resumen.to_string(index=False))

print("\n📊 Estado final de la BD:")
for tabla in ['clientes','productos','ventas']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla:12s}: {n} registros")

conn.close()
print("\n🎉 ¡Laboratorio de administración DML completado!")


---
## 💬 Actividad 3 – Análisis Reflexivo

**A)** ¿Qué es una transacción y por qué son fundamentales las propiedades ACID? Explica con un ejemplo de venta.

> _Tu respuesta aquí_

**B)** ¿Cuál es la diferencia entre DELETE, TRUNCATE y DROP? ¿Por qué SQLite no tiene TRUNCATE?

> _Tu respuesta aquí_

**C)** Compara almacenamiento local, en la nube e híbrido para una empresa peruana mediana.

> _Tu respuesta aquí_

**D)** ¿Qué riesgos existen al almacenar datos sensibles sin control de acceso? ¿Cómo los prevendrías con DCL?

> _Tu respuesta aquí_


---
## ✅ Conclusiones
1. _Conclusión sobre las operaciones DML y el control transaccional_
2. _Conclusión sobre el manejo de errores con try-except-finally_
3. _Conclusión sobre la importancia de la auditoría de operaciones_


---
## 📚 Material Complementario
| Recurso | Enlace |
|---------|--------|
| SQLite DML | https://www.sqlite.org/lang_insert.html |
| Python sqlite3 | https://docs.python.org/3/library/sqlite3.html |
| SQL Transactions | https://www.w3schools.com/sql/sql_ref_commit.asp |
| NIST Database Security | https://csrc.nist.gov/publications/detail/sp/800-111/final |
